<a href="https://colab.research.google.com/github/Ranjith0805/AI-Project/blob/main/AI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install google-generativeai streamlit pyngrok -q

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Get API key from secrets
api_key = userdata.get('GEMINI_API_KEY')

# Connect to Gemini
genai.configure(api_key=api_key)

# Choose model
model = genai.GenerativeModel('gemini-2.0-flash')

print("Gemini connected successfully!")

In [ ]:
%%writefile app.py

import streamlit as st
import google.generativeai as genai
import os

# Configure Gemini
genai.configure(api_key=os.environ.get('GEMINI_API_KEY'))

# Define model
model = genai.GenerativeModel('gemini-2.0-flash')

# App title
st.title("🤖 Siri's Study Buddy")
st.subheader("Ask me anything — I'll explain it simply!")

# Initialize chat history
if "chat_history" not in st.session_state:
    st.session_state.chat_history = [
        {
            "role": "user",
            "parts": ["You are a friendly AI Study Buddy. Explain everything simply with examples. At the end always ask if they understood."]
        },
        {
            "role": "model",
            "parts": ["Got it! I'm your Study Buddy! What would you like to learn today? 😊"]
        }
    ]

# Show chat messages
for message in st.session_state.chat_history[2:]:
    if message["role"] == "user":
        st.chat_message("user").write(message["parts"][0])
    else:
        st.chat_message("assistant").write(message["parts"][0])

# Input box
user_input = st.chat_input("Ask your question here...")

if user_input:
    st.session_state.chat_history.append({
        "role": "user",
        "parts": [user_input]
    })
    response = model.generate_content(st.session_state.chat_history)
    ai_reply = response.text
    st.session_state.chat_history.append({
        "role": "model",
        "parts": [ai_reply]
    })
    st.rerun()

In [ ]:
from google.colab import userdata
import os
import subprocess
from pyngrok import ngrok

# Kill any old tunnels
ngrok.kill()

# Set tokens from secrets
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

# Start Streamlit
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=os.environ
)

# Create public link
public_url = ngrok.connect(8501)
print(f"Your app is live at: {public_url}")